## STEP 1 - 후보 도시 비교 시각화
### 후보: Las Vegas (NV) / Phoenix (AZ) / Charlotte (NC)

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import re

# 후보 도시 설정
CANDIDATES = [
    {'state': 'NV', 'city': 'Las Vegas'},
    {'state': 'AZ', 'city': 'Phoenix'},
    {'state': 'NC', 'city': 'Charlotte'},
]

TARGET_CATEGORY = 'Restaurants'
FRANCHISE_THRESHOLD = 10

print('후보 도시:', [c['city'] for c in CANDIDATES])

In [ ]:
# 브랜드명 정제 함수
def clean_name(name):
    name = name.lower().strip()
    name = re.sub(r'[^a-z0-9\s]', '', name)
    name = re.sub(r'\s+', ' ', name).strip()
    return name

# 후보 도시별 데이터 준비
city_data = {}

for c in CANDIDATES:
    city = c['city']
    state = c['state']
    
    # 도시 + 카테고리 필터
    df = business[
        (business['state'] == state) &
        (business['city'] == city) &
        (business['categories'].str.contains(TARGET_CATEGORY, na=False))
    ].copy()
    
    # 프랜차이즈 임시 판정
    df['name_clean'] = df['name'].apply(clean_name)
    brand_cnt = df.groupby('name_clean')['business_id'].count()
    fc_brands = brand_cnt[brand_cnt >= FRANCHISE_THRESHOLD].index
    df['is_franchise'] = df['name_clean'].isin(fc_brands)
    df['type'] = df['is_franchise'].map({True: '프랜차이즈', False: '독립 브랜드'})
    
    city_data[city] = df
    
    n_fc = df['is_franchise'].sum()
    n_indie = (~df['is_franchise']).sum()
    print(f'{city:12} | 전체: {len(df):,}개 | 프랜차이즈: {n_fc:,}개 ({n_fc/len(df)*100:.1f}%) | 독립: {n_indie:,}개 ({n_indie/len(df)*100:.1f}%)')

In [ ]:
# 후보 도시 비교 요약 표
summary = []
for city, df in city_data.items():
    n_fc = df['is_franchise'].sum()
    n_indie = (~df['is_franchise']).sum()
    summary.append({
        '도시': city,
        '전체 업체수': len(df),
        '프랜차이즈': n_fc,
        '독립 브랜드': n_indie,
        '프랜차이즈 비율(%)': round(n_fc / len(df) * 100, 1),
        '평균 별점': round(df['stars'].mean(), 2),
        '평균 리뷰수': round(df['review_count'].mean(), 0),
    })

summary_df = pd.DataFrame(summary)
print('=== 후보 도시 비교 요약 ===')
print(summary_df.to_string(index=False))

In [ ]:
# 도시별 밀집도 + 프랜차이즈 혼재 지도 (3개 나란히)
colors = {'프랜차이즈': '#E74C3C', '독립 브랜드': '#3498DB'}

for city, df in city_data.items():
    fig = px.scatter_mapbox(
        df,
        lat='latitude',
        lon='longitude',
        color='type',
        color_discrete_map=colors,
        hover_name='name',
        hover_data={
            'stars': True,
            'review_count': True,
            'type': True,
            'latitude': False,
            'longitude': False
        },
        zoom=11,
        height=550,
        title=f'{city} | 전체 {len(df):,}개 | 프랜차이즈 {df["is_franchise"].sum()}개({df["is_franchise"].mean()*100:.1f}%) | 독립 {(~df["is_franchise"]).sum()}개'
    )
    fig.update_layout(mapbox_style='open-street-map')
    fig.update_layout(margin={'r': 0, 't': 50, 'l': 0, 'b': 0})
    fig.update_traces(marker=dict(size=6, opacity=0.7))
    fig.show()
    print()

In [ ]:
# 도시별 히트맵 (밀집도)
for city, df in city_data.items():
    fig = px.density_mapbox(
        df,
        lat='latitude',
        lon='longitude',
        z='review_count',
        radius=20,
        zoom=11,
        height=500,
        color_continuous_scale='YlOrRd',
        title=f'{city} - 밀집도 히트맵 (리뷰수 기반)'
    )
    fig.update_layout(mapbox_style='open-street-map')
    fig.update_layout(margin={'r': 0, 't': 50, 'l': 0, 'b': 0})
    fig.show()
    print()

In [ ]:
# 도시별 카테고리 상위 15 비교
fig, axes = plt.subplots(1, 3, figsize=(18, 8))

for ax, (city, df) in zip(axes, city_data.items()):
    cats = df['categories'].dropna()\
                           .str.split(';').explode().str.strip()\
                           .value_counts().head(15)
    cats.plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(f'{city}\n상위 카테고리', fontweight='bold', fontsize=11)
    ax.set_xlabel('업체 수')
    ax.invert_yaxis()

plt.suptitle('후보 도시별 카테고리 분포 비교', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 도시별 별점 분포 비교
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (city, df) in zip(axes, city_data.items()):
    df['stars'].hist(ax=ax, bins=9, color='steelblue', edgecolor='white', alpha=0.8)
    ax.axvline(df['stars'].mean(), color='red', linestyle='--', label=f'평균 {df["stars"].mean():.2f}')
    ax.set_title(f'{city}', fontweight='bold')
    ax.set_xlabel('별점')
    ax.set_ylabel('업체 수')
    ax.legend()

plt.suptitle('후보 도시별 별점 분포 비교', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 최종 비교 요약 출력
print('=' * 70)
print('후보 도시 최종 비교')
print('=' * 70)
for _, row in summary_df.iterrows():
    print(f"""
[{row['도시']}]
  전체 업체수     : {row['전체 업체수']:,}개
  프랜차이즈      : {row['프랜차이즈']:,}개 ({row['프랜차이즈 비율(%)']:.1f}%)
  독립 브랜드     : {row['독립 브랜드']:,}개
  평균 별점       : {row['평균 별점']}
  평균 리뷰수     : {row['평균 리뷰수']:.0f}개""")
print('=' * 70)
print('지도 확인 후 최적 도시를 선정하세요.')